# Exercise 5: Regression II

# Part 1 — Analyze early-career urban borrowers with logistic regression

### Task 1.1 — Prepare the subset

Create a filtered dataset that contains only customers from early-career
urban borrowers.

## Solution

In [2]:
df_A = df[df["customer_category"] == "early_career_urban"]

### Task 1.2 — Estimate the model

Estimate a logistic regression model to predict credit default.

## Solution

In [3]:
from sklearn.linear_model import LogisticRegression

y = df_A["default"]
X = df_A.drop(columns=["default", "customer_category"])

model = LogisticRegression()
model.fit(X, y)

model.coef_
model.intercept_

/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(

array([-0.79887951])

### Task 1.3 — Interpret the model

Interpret the coefficients in substantive terms:

- Which variables increase the likelihood of default?
- Which variables decrease it?
- Which variable appears to have the strongest relationship with
  default?

## Solution

In [4]:
import pandas as pd

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values(by="coefficient", key=abs, ascending=False)

coef_df

- Positive coefficients → increase default risk
- Negative coefficients → decrease default risk
- Largest absolute coefficient → strongest relationship

### Task 1.4 — From coefficients to classification

Explain how the model moves from:

- a linear combination of predictors
- to a predicted probability
- to a classification as default / non-default

## Solution

In [5]:
# Example for one observation
z = model.intercept_[0] + (X.iloc[0] * model.coef_[0]).sum()

import numpy as np
p = 1 / (1 + np.exp(-z))
p

np.float64(0.4680398207462943)

Steps:

1.  Linear combination (log-odds)
2.  Logistic transformation → probability
3.  Threshold (e.g., 0.5) → class label

# Part 2 — Probabilities and thresholds

In [6]:
y_prob = model.predict_proba(X)[:, 1]
y_pred = model.predict(X)

### Task 2.1 — Understanding outputs

- What is the difference between `y_prob` and `y_pred`?
- Why does `predict_proba` return two columns?

## Solution

In [7]:
y_prob[:5], y_pred[:5]

(array([0.46803982, 0.61032879, 0.95665403, 0.15395479, 0.71533136]),
 array([0, 1, 1, 0, 1]))

- `y_prob`: probabilities for class 1 (default)
- `y_pred`: binary predictions based on threshold

``` python
model.predict_proba(X[:5])
```

- Column 0 → P(no default)
- Column 1 → P(default)

### Task 2.2 — Manual thresholding

In [8]:
y_pred_03 = (y_prob >= 0.3).astype(int)

- Compare `y_pred` and `y_pred_03`
- What changes when the threshold is lowered from 0.5 to 0.3?
- Which kinds of observations are most likely to switch class?

## Solution

In [9]:
import numpy as np

np.sum(y_pred != y_pred_03)

np.int64(51)

- More observations classified as default

``` python
changed = (y_prob >= 0.3) & (y_prob < 0.5)
y_prob[changed][:5]
```

- Observations with probabilities between 0.3 and 0.5 switch class

# Part 3 — Confusion matrix and metrics

In [10]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, y_pred)
cm

array([[94, 19],
       [36, 44]])

### Task 3.1 — Interpret the matrix

Identify:

- true positives
- true negatives
- false positives
- false negatives

## Solution

In [11]:
tn, fp, fn, tp = cm.ravel()
tn, fp, fn, tp

(np.int64(94), np.int64(19), np.int64(36), np.int64(44))

- TP: predicted 1 & actual 1
- TN: predicted 0 & actual 0
- FP: predicted 1 but actual 0
- FN: predicted 0 but actual 1

### Task 3.2 — Compute key metrics

Compute and interpret:

- Accuracy
- Precision
- Recall

## Solution

In [12]:
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)

accuracy, precision, recall

(np.float64(0.7150259067357513),
 np.float64(0.6984126984126984),
 np.float64(0.55))

### Task 3.3 — Strategic reflection

LendWise is currently trying to grow its customer base aggressively.

Discuss:

- Why might the company temporarily accept a higher false-negative risk?
- Why might the same company later tighten the threshold when
  macroeconomic conditions worsen?
- What does this tell us about classification as a business decision
  rather than just a technical exercise?

## Solution

- Growth phase:

  - Lower threshold → more approvals → higher growth
  - Accept more defaults as trade-off

- Downturn:

  - Higher threshold → fewer risky loans
  - Focus on risk control

- Key insight:

  - Threshold is a **strategic lever**, not purely technical

# Part 4 — ROC curve and AUC for early-career urban borrowers

In [13]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y, y_prob)
roc_auc = auc(fpr, tpr)

### Task 4.1 — Understanding ROC

- What does the ROC curve represent?
- What are TPR and FPR?

## Solution

In [14]:
fpr[:5], tpr[:5]

(array([0.        , 0.        , 0.        , 0.00884956, 0.00884956]),
 array([0.    , 0.0125, 0.2375, 0.2375, 0.25  ]))

- ROC: trade-off between TPR and FPR across thresholds

- TPR = TP / (TP + FN)

- FPR = FP / (FP + TN)

### Task 4.2 — Interpreting AUC

- What does the AUC value tell us?
- What would values near 0.5, 0.7, and 1.0 suggest?

## Solution

In [15]:
roc_auc

0.7972345132743363

- 0.5 → random
- ~0.7 → moderate
- 1.0 → perfect

### Task 4.3 — Threshold perspective

- Why is ROC/AUC useful when the company has not yet fixed one single
  decision threshold?

## Solution

- Evaluates performance across all thresholds
- Supports flexible decision-making
- Enables model comparison independent of cutoff

# Part 5 — Compare early-career urban borrowers and early-career urban borrowers

Now repeat the same modeling steps the segment of platform workers.

### Task 5.1 — Fit the second model

Fit a logistic regression model for early-career urban borrowers.

## Solution

In [16]:
df_B = df[df["customer_category"] == "platform_worker"]

y_B = df_B["default"]
X_B = df_B.drop(columns=["default", "customer_category"])

model_B = LogisticRegression()
model_B.fit(X_B, y_B)

/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(

### Task 5.2 — Compare coefficients

Compare the coefficients across both categories.

- Are the same predictors important in both groups?
- Does any variable appear to matter more strongly in one group than the
  other?

## Solution

In [17]:
coef_A = pd.Series(model.coef_[0], index=X.columns)
coef_B = pd.Series(model_B.coef_[0], index=X_B.columns)

comparison = pd.DataFrame({
    "Category_A": coef_A,
    "Category_B": coef_B
})

comparison

### Task 5.3 — Compare ROC and AUC

Compare the ROC curves and AUC values of the two models.

- Which category seems easier to classify?
- What might explain the difference?

## Solution

In [18]:
y_prob_B = model_B.predict_proba(X_B)[:, 1]

fpr_B, tpr_B, _ = roc_curve(y_B, y_prob_B)
roc_auc_B = auc(fpr_B, tpr_B)

roc_auc, roc_auc_B

(0.7972345132743363, 0.7174603174603175)

- Higher AUC → easier classification

### Task 5.4 — Interpretation challenge

Suppose the AUC for early-career urban borrowers is much higher than for
early-career urban borrowers.

Discuss at least two possible explanations, such as:

- the predictors are more informative in one group
- one group is more heterogeneous
- the same variables do not capture risk equally well in both customer
  categories
- additional predictors may be needed for one category

## Solution

Possible explanations:

- Predictors are more informative in early-career urban borrowers
- The group of early-career urban borrowers is more heterogeneous
- Missing variables for early-career urban borrowers
- Different behavioral patterns

In [19]:
comparison.abs().sort_values(by="Category_A", ascending=False)

- Differences in coefficient magnitude support this reasoning

# 🧩 Wrap-up <a id="wrap-up"></a>

🎉🎈 You have completed the notebook – good work! 🎈🎉

In this notebook, we have learned to:

- Fit and interpret logistic regression models using `scikit-learn`
- Understand how coefficients relate to default risk in a realistic
  fintech setting
- Distinguish between predicted probabilities and class labels
- Apply and reflect on different classification thresholds
- Compute and interpret confusion-matrix-based metrics
- Understand ROC curves and AUC as threshold-independent evaluation
  tools
- Compare model performance across different customer categories
- Reflect on how model use depends on business context and risk strategy

> **Session 5 survey**
>
> Before you wrap up, please complete the Session 5 survey here:
> <a href="https://forms.gle/mVWw3z7ftFn48gDY6"
> target="_blank">https://forms.gle/mVWw3z7ftFn48gDY6</a>. Thank you 🙏